# Part 2-4. 센서 데이터 로그 분석 및 가공

드론 한 대가 5분 동안 비행하면서 남긴 **로그 두 개**를 분석한다.
같은 비행을 두 시스템이 각자 기록했다.

| 파일 | 어디서 나온 것 | 시각 기준 | 좌표계 |
|---|---|---|---|
| `flight.mcap` | ROS2 (rosbag2) | 1970년 기준 나노초 | ENU (동-북-상) |
| `flight.ulg` | PX4 비행제어 소프트웨어 | 부팅 후 마이크로초 | NED (북-동-하) |

**진행 방식** — 과제 1부터 7까지 순서대로 푼다. 각 과제의 코드 셀에서 `# IMPLEMENT HERE` 자리를 채운다.
바로 앞 과제의 결과를 다음 과제가 그대로 쓴다. 셀은 위에서 아래로 한 번씩 실행한다.

**쓰는 도구** — Part 1(모듈과 패키지), Part 2-1(Matplotlib), 2-2(NumPy), 2-3(Pandas)에서 배운 것만 쓴다.
로그 파일을 여는 코드는 `loaders.py` 로 주어진다. 직접 만들 필요 없다.

**제출물** — 채운 이 노트북, `out/clean.csv`, `out/segments.csv`, `out/dashboard.png`, `flightkit/` 폴더, 그리고 짧은 `report.md`.

In [ ]:
# 실습 준비. 이 셀만 먼저 실행한다 (Colab 기준 30초 정도)
!pip -q install mcap mcap-ros2-support pyulog
!wget -q https://stmoon.github.io/python-lecture-slides/part2-4-data/flight.mcap -O flight.mcap
!wget -q https://stmoon.github.io/python-lecture-slides/part2-4-data/flight.ulg -O flight.ulg
!wget -q https://stmoon.github.io/python-lecture-slides/part2-4-data/loaders.py -O loaders.py

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from loaders import load_mcap, load_ulog, describe

matplotlib.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 120)

import os
os.makedirs("out", exist_ok=True)
print("준비 완료:", os.path.getsize("flight.mcap"), "bytes /", os.path.getsize("flight.ulg"), "bytes")

## 0. 배경 (10분)

### ROS2 와 mcap

**ROS2** 는 로봇 소프트웨어를 여러 **노드**로 나누고, 노드끼리 **토픽**으로 메시지를 주고받게 하는 미들웨어다.
카메라 노드가 `/image` 토픽으로 사진을 흘리면, 인식 노드가 그것을 받아 쓰는 식이다.

**mcap** 은 그 토픽들을 시간순으로 담아 두는 기록 파일 형식이다 (ROS2 의 `rosbag2` 가 쓰는 기본 형식).
파일 하나 안에 여러 토픽이 들어 있고, 토픽마다 메시지 구조(스키마)가 다르다.
이번 파일에는 네 토픽이 있다.

| 토픽 | 메시지 타입 | 주기 | 내용 |
|---|---|---|---|
| `/imu/data` | `sensor_msgs/msg/Imu` | 25 Hz | 가속도, 각속도, 자세 |
| `/fix` | `sensor_msgs/msg/NavSatFix` | 5 Hz | 위도, 경도, 고도 |
| `/cmd_vel` | `geometry_msgs/msg/Twist` | 10 Hz | 명령 속도 |
| `/temperature` | `sensor_msgs/msg/Temperature` | 1 Hz | 보드 온도 |

### PX4 와 ulog

**PX4** 는 드론의 비행제어 소프트웨어다. 자세를 추정하고 모터 명령을 계산하는 일을 한다.
내부 모듈끼리는 **uORB** 라는 메시지 버스로 값을 주고받고, 그 메시지들을 그대로 파일에 남긴 것이 **ulog** 다.

| 메시지 | 주기 | 내용 |
|---|---|---|
| `sensor_combined` | 50 Hz | 가속도 3축, z축 각속도 |
| `vehicle_local_position` | 20 Hz | 위치와 속도 (NED) |
| `vehicle_attitude` | 20 Hz | roll, pitch, yaw |
| `battery_status` | 2 Hz | 전압, 전류, 잔량 |

### 두 로그의 공통점

이름과 포맷은 다르지만 구조는 같다. **"시각 열 하나 + 값 열 여러 개"인 표가 여러 개** 있는 것이다.
열고 나면 둘 다 DataFrame 이고, 그 다음부터는 Pandas 로 똑같이 다룬다.

주의할 차이 세 가지.

1. **시각 기준이 다르다** — mcap 은 1970년 기준 나노초, ulog 은 부팅 후 마이크로초. 그대로 비교하면 안 된다.
2. **좌표계가 다르다** — ROS2 는 ENU(동-북-상), PX4 는 NED(북-동-하). PX4 의 `z` 는 아래가 양수다.
3. **주기가 다르다** — 25 Hz 와 50 Hz 를 나란히 놓으려면 같은 주기로 다시 샘플링해야 한다.

## 과제 1. 로그 열고 구조 파악 (15분)

두 로그를 열고 무엇이 들어 있는지 확인한다.

**할 일**
1. `load_mcap("flight.mcap")` 과 `load_ulog("flight.ulg")` 로 표를 읽는다.
2. `describe()` 로 표 목록과 열 이름을 출력한다.
3. `/imu/data` 를 `imu`, `/fix` 를 `fix`, `sensor_combined` 를 `sc`,
   `vehicle_local_position` 을 `lp`, `vehicle_attitude` 를 `att` 에 담는다.
4. `imu` 의 `head()`, `info()`, `describe()` 를 확인한다.
5. 시각 열의 차이(`diff`)의 중앙값으로 두 IMU 의 **표본 주기(Hz)** 를 각각 계산한다.

**확인 항목** — mcap 4개, ulog 4개 표. `/imu/data` 는 7500행, `sensor_combined` 는 15000행.
주기는 각각 25 Hz, 50 Hz 근처.

**흔한 실수** — `t_ns` 는 나노초, `timestamp` 는 마이크로초다. 나누는 수가 다르다.

In [ ]:
topics = load_mcap("flight.mcap")
msgs = load_ulog("flight.ulg")

describe(topics)
describe(msgs)

# IMPLEMENT HERE

## 과제 2. 시간축 정리 (15분)

두 로그의 시각을 **같은 기준의 경과 초**로 바꾼다.

**할 일**
1. mcap 전체에서 가장 이른 `t_ns` 를 `t0_ns` 로 잡는다. `imu`, `fix` 에 `t = (t_ns - t0_ns) / 1e9` 열을 만든다.
2. ulog 전체에서 가장 이른 `timestamp` 를 `t0_us` 로 잡는다. `sc`, `lp`, `att` 에 `t = (timestamp - t0_us) / 1e6` 열을 만든다.
3. `imu` 에 중복된 시각이 있다. 중복을 제거하고 시각 순으로 정렬한다 (인덱스도 다시 매긴다).
4. `fix` 의 시각 간격(`diff`)을 구해 **정상 간격보다 크게 벌어진 구간**을 찾아 출력한다.

**확인 항목** — 중복 제거 후 `imu` 는 7497행. `fix` 에서 15초짜리 구멍 하나가 나온다.

**흔한 실수** — `drop_duplicates` 는 기본이 모든 열 기준이다. 시각 열을 지정해야 한다.
정렬 뒤 `reset_index(drop=True)` 를 빠뜨리면 뒤 과제에서 인덱스가 어긋난다.

In [ ]:
t0_ns = min(df["t_ns"].min() for df in topics.values())
t0_us = min(df["timestamp"].min() for df in msgs.values())

# IMPLEMENT HERE

## 과제 3. 결측과 이상치 (20분)

값이 이상한 자리를 골라낸다. **물리적으로 불가능한 값을 먼저 지우고**, 그 다음에 통계로 본다.

**할 일**
1. `fix["altitude"]` 에 센티널 `-999.0` 이 섞여 있다. 결측(`NaN`)으로 바꾸고 결측 비율을 출력한다.
2. 바뀐 결측을 앞뒤 값으로 보간한다 (`interpolate`).
3. `sc["accelerometer_m_s2_z"]` 에서 **크기가 40 m/s^2 를 넘는 값**을 결측으로 바꿔 `az` 열을 만든다.
   중력이 9.8 이므로 40 을 넘는 값은 센서 오류다.
4. 만든 결측을 보간한다.
5. 보간 후 `az` 의 최댓값을 확인한다. 20 을 넘는 값 두 개가 남는데, 이것은 **오류가 아니라 실제 충격**이다.
   몇 초에 있었는지 출력한다.

**확인 항목** — altitude 결측 3개. `az` 에서 지워지는 값 12개. 남는 충격 2회.

**흔한 실수** — z-score 로 먼저 자르면 실제 충격까지 지워진다. 물리 범위가 먼저다.
`replace(-999, np.nan)` 은 새 Series 를 돌려준다. 열에 다시 대입해야 반영된다.

In [ ]:
# IMPLEMENT HERE

## 과제 4. 단위와 좌표 (15분)

두 로그의 좌표계를 맞추고, 센서 장착 오차를 보정한다.

**할 일**
1. `lp` 는 PX4 의 **NED** 좌표다. `x` 가 북, `y` 가 동, `z` 는 **아래가 양수**다.
   ROS2 와 같은 **ENU** 로 바꿔 `east`, `north`, `up` 열을 만든다.
2. `up` 의 최댓값으로 최고 고도를 확인한다.
3. IMU 가 기체 앞방향에서 **15도 틀어져** 장착되어 있다. 2차원 회전 행렬을 만들어
   `imu` 의 `linear_acceleration_x`, `linear_acceleration_y` 를 기체 좌표로 돌려
   `ax_body`, `ay_body` 열에 넣는다.
4. 보정 전후의 x축 가속도 평균을 비교한다.

**회전 행렬** — 각도 `th` 에 대해

```
R = [[cos(th), -sin(th)],
     [sin(th),  cos(th)]]
```

점을 행으로 쌓은 배열 `P` 에는 `P @ R.T` 로 적용한다 (NumPy 연습문제 9 와 같은 방식).

**확인 항목** — 최고 고도 30 m 근처. 보정 후 값이 달라진다.

**흔한 실수** — `up = -z` 다. 부호를 빠뜨리면 고도가 음수로 나온다.
`np.deg2rad` 없이 15 를 그대로 넣으면 라디안이 아니라 15 라디안이 된다.

In [ ]:
# IMPLEMENT HERE

## 과제 5. 두 로그 합치기 (20분)

주기가 다른 두 로그를 **같은 시간 격자**에 올려 나란히 본다.

**할 일**
1. `imu` 와 `sc` 의 `t`(경과 초)를 `pd.to_datetime(..., unit="s")` 로 바꿔 인덱스로 세운다.
2. 각각 **1초 평균**으로 리샘플링한다. mcap 쪽은 `linear_acceleration_z`, ulog 쪽은 `az` 를 쓴다.
3. 두 결과를 시간 인덱스 기준으로 결합해 `merged` 를 만든다. 열 이름은 `ros_az`, `px4_az`.
4. 차이 열 `diff = ros_az - px4_az` 를 만들고 `describe()` 로 요약한다.
5. `lp` 의 `up` 도 같은 방식으로 1초 평균을 내어 `merged` 에 `up` 열로 붙인다.
6. `merged` 를 `out/clean.csv` 로 저장한다.

**확인 항목** — `merged` 는 300행 근처. 두 가속도의 차이는 평균 0 근처에서 작게 흔들린다.

**흔한 실수** — 리샘플링은 **시간 인덱스**가 있어야 동작한다. `t` 열만 있고 인덱스가 정수면 오류가 난다.
`merge` 는 인덱스로 붙일 때 `left_index=True, right_index=True` 가 필요하다.

In [ ]:
# IMPLEMENT HERE

## 과제 6. 집계와 시각화 (20분)

비행 구간을 나누고, 구간별 통계와 그림을 만든다.

**할 일**
1. `merged` 에서 `up > 1.0` 이면 비행 중으로 본다. 불리언 열 `flying` 을 만든다.
2. 상태가 바뀔 때마다 번호가 올라가는 구간 라벨을 만든다.
   `(flying != flying.shift()).cumsum()` 이 그 번호다. `seg` 열에 넣는다.
3. `seg` 로 묶어 구간별 **시작 시각(초), 길이(초), 평균 고도, 최대 가속도, 비행 여부**를 담은 표를 만든다.
   `out/segments.csv` 로 저장한다.
4. 2행 2열 대시보드를 그려 `out/dashboard.png` 로 저장한다.
   - 좌상: 시간에 따른 고도 `up`
   - 우상: `ros_az` 와 `px4_az` 두 선 (범례 포함)
   - 좌하: `diff` 히스토그램 (bins=30)
   - 우하: `ros_az` 와 `px4_az` 의 산점도

**확인 항목** — 구간은 3개 (지상 - 비행 - 지상). 비행 구간 길이는 250초 근처.

**흔한 실수** — `groupby` 결과의 열 이름은 `agg(이름="함수")` 로 직접 정하는 편이 읽기 좋다.
`savefig` 는 `show` 보다 **먼저** 불러야 한다. 그린 다음 비워지기 때문이다.

In [ ]:
# IMPLEMENT HERE

## 과제 7. 패키지로 묶기 (15분)

여기까지 쓴 처리를 **다음 로그에도 그대로 쓸 수 있게** 패키지로 만든다. Part 1 에서 배운 구조다.

**할 일**
1. `flightkit/` 폴더를 만든다.
2. `%%writefile` 로 `flightkit/clean.py` 를 쓴다. 함수 두 개를 넣는다.
   - `to_seconds(df, col, t0, scale)` — 시각 열을 경과 초 `t` 열로 바꿔 돌려준다
   - `drop_out_of_range(s, lo, hi)` — 범위 밖 값을 결측으로 바꾸고 보간해 돌려준다
3. `%%writefile` 로 `flightkit/__init__.py` 를 쓴다. 두 함수를 재노출하고 `__all__` 을 정한다.
4. `import flightkit` 후 `flightkit.drop_out_of_range` 로 과제 3 을 다시 한 번 돌려
   결과가 같은지 확인한다.

**확인 항목** — `flightkit.__all__` 이 두 이름을 담는다. 다시 돌린 결과의 최댓값이 과제 3 과 같다.

**흔한 실수** — Colab 에서 파일을 고친 뒤 다시 `import` 해도 예전 것이 그대로 쓰인다 (모듈 캐시).
`importlib.reload` 를 쓰거나 셀을 처음부터 다시 실행한다.

In [ ]:
os.makedirs("flightkit", exist_ok=True)

# 아래 두 셀의 %%writefile 을 채운 뒤 마지막 셀에서 import 해 확인한다
# IMPLEMENT HERE

## 마무리

`report.md` 에 아래 네 가지를 각각 두세 줄로 적는다.

1. 두 로그에서 발견한 문제 (몇 개, 어디서)
2. 각 문제를 어떻게 처리했고 왜 그렇게 했는지
3. 두 로그의 가속도 값이 얼마나 일치했는지
4. 이 파이프라인을 다른 비행 로그에 쓸 때 먼저 확인해야 할 것

**제출물** — 이 노트북, `out/clean.csv`, `out/segments.csv`, `out/dashboard.png`, `flightkit/`, `report.md`